In [164]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.lower().endswith(".csv"):
            print(os.path.join(root, file))

/kaggle/input/datasets/mdtaslimulmahbubjitu/bbc-text/bbc-text-preprocessed.csv


DATASET 2 - BBC NEWS CATEGORIES

In [165]:
import os
import glob
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

1. CONFIGURATION

In [166]:
MODEL_NAME = "bert-base-uncased"

LEARNING_RATE = 0.00003
BATCH_SIZE = 16
WEIGHT_DECAY = 0.1

EPOCHS = 5
WARMUP_RATIO = 0.1

MAX_LEN = 128

DROPOUT = 0.1

SEED = 42

2. DATASET PATH

In [167]:

FILE_NAME = "bbc-text-preprocessed.csv"


file_matches = glob.glob(
    f"/kaggle/input/**/{FILE_NAME}",
    recursive=True
)

if len(file_matches) == 0:
    raise FileNotFoundError(
        f"Could not find {FILE_NAME} inside /kaggle/input/"
    )

NEWS_FILE = file_matches[0]

print("Dataset path:", NEWS_FILE)



Dataset path: /kaggle/input/datasets/mdtaslimulmahbubjitu/bbc-text/bbc-text-preprocessed.csv


3. DEVICE

In [168]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


Device: cuda
GPU: Tesla T4


4. SEED

In [169]:

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


5. DATASET CLASS

In [170]:
class TextDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_len=MAX_LEN
    ):

        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding="max_length",
            max_length=max_len
        )

        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(
                value[idx],
                dtype=torch.long
            )
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item



6. METRICS

In [171]:

def compute_metrics(pred):

    labels = pred.label_ids
    predictions = pred.predictions

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    preds = np.argmax(
        predictions,
        axis=-1
    )

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            labels,
            preds,
            average="weighted",
            zero_division=0
        )
    )

    accuracy = accuracy_score(
        labels,
        preds
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }



7. LOAD DATASET

In [172]:
print("\n" + "=" * 70)
print("Running BERT on: Dataset2_BBCNewsCategories")
print("=" * 70)

df = pd.read_csv(NEWS_FILE)

print("Original shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())




Running BERT on: Dataset2_BBCNewsCategories
Original shape: (2225, 2)

Columns:
['category', 'clean_text']


8. DATA CLEANING

In [173]:

required_columns = [
    "clean_text",
    "category"
]

missing_columns = [
    col
    for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )



df = df.dropna(
    subset=[
        "clean_text",
        "category"
    ]
).copy()



df["clean_text"] = (
    df["clean_text"]
    .astype(str)
    .str.strip()
)

df["category"] = (
    df["category"]
    .astype(str)
    .str.strip()
    .str.lower()
)



df = df[
    (df["clean_text"] != "") &
    (df["category"] != "")
].reset_index(drop=True)

print(
    "After removing missing/empty values:",
    df.shape
)



before_duplicates = len(df)

df = df.drop_duplicates(
    subset=["clean_text"]
).reset_index(drop=True)

removed_duplicates = (
    before_duplicates - len(df)
)

print(
    "Duplicate rows removed:",
    removed_duplicates
)

print(
    "Final dataset shape:",
    df.shape
)



After removing missing/empty values: (2225, 2)
Duplicate rows removed: 108
Final dataset shape: (2117, 2)


LABEL ENCODING

In [174]:
labels_unique = sorted(
    df["category"].unique()
)

label2id = {
    label: idx
    for idx, label in enumerate(labels_unique)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

df["label_id"] = df["category"].map(
    label2id
)

num_labels = len(labels_unique)

print(
    "\nNumber of classes:",
    num_labels
)

print(
    "Classes:",
    labels_unique
)

print(
    "\nClass distribution:"
)

print(
    df["category"].value_counts()
)

print(
    "\nLabel mapping:",
    label2id
)



Number of classes: 5
Classes: ['business', 'entertainment', 'politics', 'sport', 'tech']

Class distribution:
category
business         503
sport            501
politics         403
entertainment    369
tech             341
Name: count, dtype: int64

Label mapping: {'business': 0, 'entertainment': 1, 'politics': 2, 'sport': 3, 'tech': 4}


10. TRAIN / VALIDATION / TEST SPLIT

In [175]:
train_texts, temp_texts, train_labels, temp_labels = (
    train_test_split(
        df["clean_text"].tolist(),
        df["label_id"].tolist(),
        test_size=0.20,
        random_state=SEED,
        stratify=df["label_id"]
    )
)

val_texts, test_texts, val_labels, test_labels = (
    train_test_split(
        temp_texts,
        temp_labels,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_labels
    )
)


print(
    "\nTraining samples:",
    len(train_texts)
)

print(
    "Validation samples:",
    len(val_texts)
)

print(
    "Testing samples:",
    len(test_texts)
)


Training samples: 1693
Validation samples: 212
Testing samples: 212


11. TOKENIZER

In [176]:
tokenizer = BertTokenizerFast.from_pretrained(
    MODEL_NAME
)



12. CREATE DATASETS

In [177]:
train_dataset = TextDataset(
    train_texts,
    train_labels,
    tokenizer,
    MAX_LEN
)

val_dataset = TextDataset(
    val_texts,
    val_labels,
    tokenizer,
    MAX_LEN
)

test_dataset = TextDataset(
    test_texts,
    test_labels,
    tokenizer,
    MAX_LEN
)


13. BERT MODEL

In [178]:
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=DROPOUT,
    attention_probs_dropout_prob=DROPOUT
)

model.to(device)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

14. TRAINING ARGUMENTS

In [179]:
training_args = TrainingArguments(

    output_dir="/kaggle/working/Dataset2_BBCNewsCategories",

    num_train_epochs=EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,

    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,

    warmup_ratio=WARMUP_RATIO,

   
    eval_strategy="epoch",

   
    save_strategy="epoch",
    save_total_limit=1,

   
    load_best_model_at_end=True,

    metric_for_best_model="f1",
    greater_is_better=True,

    logging_steps=20,

    optim="adamw_torch",

    seed=SEED,
    data_seed=SEED,

    report_to="none",

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=2
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


15. TRAINER

In [180]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

 
    eval_dataset=val_dataset,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)



16. TRAINING

In [181]:
print("\nStarting training...")

train_result = trainer.train()



Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,2.023603,0.545823,0.933962,0.939675,0.933962,0.933008
2,0.317952,0.172157,0.976415,0.976347,0.976415,0.976327
3,0.090016,0.246113,0.976415,0.976817,0.976415,0.976420
4,0.048683,0.325778,0.971698,0.972548,0.971698,0.971680
5,0.044991,0.335822,0.971698,0.972548,0.971698,0.971680


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

17. FINAL TEST EVALUATIO

In [182]:
print("\nEvaluating on unseen test set...")

test_metrics = trainer.evaluate(
    eval_dataset=test_dataset,
    metric_key_prefix="test"
)


Evaluating on unseen test set...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


early stopping required metric_for_best_model, but did not find eval_f1 so early stopping is disabled


18. FINAL RESULTS

In [183]:
print("\n" + "=" * 70)
print("DATASET 2 - BBC NEWS CATEGORIES RESULTS")
print("=" * 70)

print(
    f"Accuracy  : "
    f"{test_metrics['test_accuracy'] * 100:.2f}%"
)

print(
    f"Precision : "
    f"{test_metrics['test_precision'] * 100:.2f}%"
)

print(
    f"Recall    : "
    f"{test_metrics['test_recall'] * 100:.2f}%"
)

print(
    f"F1 Score  : "
    f"{test_metrics['test_f1'] * 100:.2f}%"
)

print("=" * 70)


DATASET 2 - BBC NEWS CATEGORIES RESULTS
Accuracy  : 98.58%
Precision : 98.58%
Recall    : 98.58%
F1 Score  : 98.58%
